# Pipeline 1 — FerretNet (LPD-Based Forensics)

This notebook tests the **FerretNet** pipeline for AI vs Real image detection.  
FerretNet is a Learned Pixel Difference (LPD) forensics model (NeurIPS 2025) with a dual-branch architecture: one branch for face crops and one for full images.  
It was pretrained on the ForenSynths dataset (GAN-era images) and fine-tuned on **GRAVEX-200K** (200k images).  

**Spoiler:** FerretNet achieved ~50% accuracy — barely better than random — because the GAN-specific features it learned do not transfer to modern diffusion-based AI images.

## 1. Setup & Imports

In [ ]:
import sys, os, random, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Project path setup ──
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Shared notebook utilities ──
import nb_utils

# ── FerretNet inference pipeline ──
from inference.ferretnet_pipeline import FerretNetPipeline

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Random seed:  {SEED}")

## 2. Dataset Overview (GRAVEX-200K)

**GRAVEX-200K** is the dataset used for training and evaluation:

| Property | Value |
|----------|-------|
| Total images | 200,000 |
| Real images | 100,000 (50%) |
| AI-generated images | 100,000 (50%) |
| Split ratio | 70% train / 20% val / 10% test |
| Real sources | FaceForensics++, DFDC, CelebA-HQ, FFHQ |
| AI sources | Stable Diffusion faces, StyleGAN, Midjourney, DALL-E |

The dataset was pre-split by the original authors to prevent data leakage. Face crops were extracted using RetinaFace during preprocessing.

In [ ]:
samples = nb_utils.show_dataset_overview(n_samples=8)

## 3. Preprocessing Pipeline

FerretNet uses a 3-step preprocessing pipeline before inference:

1. **Resize with interpolation** — Images are resized to 256x256 using Lanczos interpolation, which preserves fine details better than bilinear/bicubic for forensic analysis.
2. **Color space analysis** — RGB channels are analyzed individually. AI-generated images often show subtle inconsistencies in individual color channels that are invisible in the composite view.
3. **Noise reduction** — Gaussian, median, and bilateral filters are compared. The choice of denoising affects which forensic artifacts survive — aggressive filtering can destroy the very pixel-level differences that FerretNet's LPD layers try to detect.

In [ ]:
# Pick a sample image from the test set
sample_image_path = samples[0]["image_path"]
print(f"Sample image: {Path(sample_image_path).name}")

nb_utils.show_preprocessing_steps(sample_image_path, target_size=256)

## 4. Augmentation Study

Data augmentation is critical for training a robust forensics detector. Each augmentation addresses a specific failure mode:

- **Geometric transforms** (rotation, flip, crop, translation) — prevent the model from memorizing spatial positions and force invariance to framing.
- **Blur / Sharpening** — simulate real-world camera conditions and social media processing; the model must detect AI artifacts even through defocus.
- **Color jitter** — prevents learning brightness or white balance as a proxy for the real/AI label.
- **JPEG compression** — simulates the lossy re-encoding that happens on social media platforms; prevents the model from relying on compression artifacts rather than semantic AI fingerprints.

**Key failure mode:** Over-augmentation can destroy the very pixel-level artifacts that FerretNet relies on. The augmentation probabilities were tuned to balance diversity with artifact preservation.

In [ ]:
nb_utils.show_augmentation_study(sample_image_path, size=256)

## 5. Model Architecture — FerretNet

FerretNet uses **Learned Pixel Difference (LPD)** convolutions to capture local forensic artifacts at the pixel level.

### Architecture Details

| Component | Details |
|-----------|--------|
| Core module | `Ferret` (from `ferretnet/ferret.py`) |
| LPD function | Median (3x3 window) |
| Embedding dim | 96 |
| Depths | [2, 2] |
| Parameters per branch | ~1.06M |
| Branches | Face branch + Full image branch |
| Wrapper | `DualBranchFerretNet` (`ferretnet/dual_branch.py`) |
| Fusion | 0.6 * face_score + 0.4 * full_score |
| Input size | 256 x 256 |
| Pretrained on | ForenSynths (GAN-generated images) |
| Fine-tuned on | GRAVEX-200K |

The dual-branch design processes face crops and full images independently, then fuses scores with a weighted average. When no face is detected, the full-branch score is used alone.

In [ ]:
from ferretnet.dual_branch import DualBranchFerretNet
from configs.base_config import ProjectConfig
import torch

proj = ProjectConfig()
model = DualBranchFerretNet(pretrained_path=None)  # just show architecture

total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")
print(f"Face branch: {sum(p.numel() for p in model.face_net.parameters()):,}")
print(f"Full branch: {sum(p.numel() for p in model.full_net.parameters()):,}")
print(f"\nFusion config: face_weight={model.fusion.face_weight}, full_weight={model.fusion.full_weight}")
print(f"Device: {'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'}")

## 6. Training Configuration

FerretNet was fine-tuned on GRAVEX-200K with the following configuration:

| Parameter | Value |
|-----------|-------|
| Epochs | 15 |
| Batch size | 32 |
| Learning rate | 1e-4 |
| Optimizer | Adam |
| Scheduler | CosineAnnealing |
| Loss function | BCEWithLogitsLoss |
| Early stopping patience | 5 |
| Precision | 16-mixed (AMP) |
| Normalization mean | [0.48145466, 0.4578275, 0.40821073] |
| Normalization std | [0.26862954, 0.26130258, 0.27577711] |

In [ ]:
import pandas as pd

val_csv = PROJECT_ROOT / "results" / "val_metrics.csv"

if val_csv.exists():
    df = pd.read_csv(val_csv)
    print(f"Training log: {len(df)} rows")
    print(df.head(10))

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    ax = axes[0]
    if "train_loss" in df.columns:
        valid = df.dropna(subset=["train_loss"])
        ax.plot(valid["epoch"], valid["train_loss"], label="Train Loss", marker="o", markersize=3)
    if "val_loss" in df.columns:
        valid = df.dropna(subset=["val_loss"])
        ax.plot(valid["epoch"], valid["val_loss"], label="Val Loss", marker="s", markersize=3)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("FerretNet — Training & Validation Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Accuracy curves
    ax = axes[1]
    for col, label in [("val_face_acc", "Face Acc"), ("val_full_acc", "Full Acc"), ("val_fused_acc", "Fused Acc")]:
        if col in df.columns:
            valid = df.dropna(subset=[col])
            if len(valid) > 0:
                ax.plot(valid["epoch"], valid[col], label=label, marker="o", markersize=3)
    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Random baseline")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("FerretNet — Validation Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No val_metrics.csv found — training curves unavailable.")

## 7. Sample Predictions

In [ ]:
pipeline = FerretNetPipeline()

if pipeline.is_available():
    print(f"Pipeline: {pipeline.name}")
    print(f"Description: {pipeline.description}")
    print(f"Checkpoint found — running predictions...\n")
    nb_utils.show_predictions(pipeline, samples, n=8)
else:
    print("FerretNet checkpoint not found. Train the model first:")
    print("  python training/train.py")

## 8. Evaluation Metrics

In [ ]:
results_json = PROJECT_ROOT / "results" / "evaluation" / "results.json"

if results_json.exists():
    with open(results_json) as f:
        results = json.load(f)

    print("=" * 60)
    print("FerretNet — Evaluation Results (from results.json)")
    print("=" * 60)

    for branch_name in ["face_branch", "full_branch", "fused"]:
        if branch_name in results:
            b = results[branch_name]
            print(f"\n  {branch_name.upper()}")
            print(f"  Accuracy:  {b['accuracy']:.4f} ({b['accuracy']:.1%})")
            print(f"  AUC-ROC:   {b['auc_roc']:.4f}")
            print(f"  Avg Prec:  {b['average_precision']:.4f}")
            print(f"  EER:       {b['eer']:.4f}")
            if "classification_report" in b:
                print(f"\n{b['classification_report']}")

elif pipeline.is_available():
    print("No saved results found — running live evaluation...")
    y_true, y_scores = nb_utils.evaluate_pipeline(pipeline, samples, max_images=2000)
    nb_utils.show_metrics(y_true, y_scores, model_name="FerretNet", threshold=0.5)

else:
    print("No results.json found and no checkpoint available.")
    print("Train the model first, then run evaluation.")

In [ ]:
# If we have both a live pipeline and saved results, show the visual metrics
if pipeline.is_available() and results_json.exists():
    print("Running live evaluation for confusion matrix and ROC curve...")
    y_true, y_scores = nb_utils.evaluate_pipeline(pipeline, samples, max_images=2000)
    nb_utils.show_metrics(y_true, y_scores, model_name="FerretNet", threshold=0.5)

## 9. Inference Speed

In [ ]:
if pipeline.is_available():
    nb_utils.show_inference_speed(pipeline, samples, n_runs=20)
else:
    print("FerretNet checkpoint not available — skipping speed benchmark.")

## 10. Summary

### FerretNet Results

| Metric | Face Branch | Full Branch | Fused |
|--------|-------------|-------------|-------|
| Accuracy | 50.6% | 50.4% | 50.4% |
| AUC-ROC | 0.478 | 0.471 | 0.471 |
| EER | 0.514 | 0.521 | 0.530 |

### Verdict: POOR (~50% accuracy, barely above random chance)

### Root Cause Analysis

FerretNet was pretrained on **ForenSynths**, a dataset of GAN-generated faces (ProGAN, StyleGAN, etc.). GANs produce characteristic artifacts:
- Spectral peaks from upsampling convolutions
- Checkerboard patterns in pixel space
- Consistent frequency-domain fingerprints

FerretNet's **LPD (Learned Pixel Difference)** layers are specifically designed to capture these local, pixel-level GAN artifacts. However, modern AI image generators (Stable Diffusion, DALL-E, Midjourney) use **diffusion-based** architectures that produce fundamentally different artifacts:
- Smooth, globally coherent textures
- Semantic inconsistencies (hands, text, reflections) rather than pixel-level patterns
- No checkerboard or spectral artifacts

The pretrained LPD features are **specialized for GAN fingerprints** and simply do not generalize to diffusion-based artifacts. Fine-tuning on GRAVEX-200K could not overcome this domain gap because the feature extractor was too deeply committed to the wrong artifact type.

### Next Steps

This motivated switching to **CLIP-based features** (see **Notebook 02**), which capture high-level semantic representations rather than low-level pixel artifacts, and are therefore more robust to the shift from GAN to diffusion generators.